# 08 — Model Comparison & Selection

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

Notebook 07 established the first out-of-sample benchmark:
- Primary target: `Target_DD_5Pct_10D`
- Chronological 70/15/15 split
- Validation-based model selection
- Validation-based alert threshold
- Final untouched test evaluation

Notebook 08 deepens the comparison before we lock the candidate model for explainability and final evaluation.

**Important:** The test set remains untouched for model/threshold decisions in this notebook.


## 1. Inputs

Required files from Notebook 06/07:
- `data/processed/portfolio_ml_features.csv`
- `reports/model_comparison.csv`
- `reports/test_predictions.csv`
- `reports/model_split_summary.csv`
- `reports/model_feature_importance.csv`

Notebook 07 also created the fitted model artifact:
- `models/07_best_model.joblib`

We will independently reproduce the chronological training/validation split here so that model-selection diagnostics are based only on development data.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

DATA = Path("../data/processed")
REPORTS = Path("../reports")
MODELS = Path("../models")

INPUT_PATH = DATA / "portfolio_ml_features.csv"
TARGET = "Target_DD_5Pct_10D"
RANDOM_STATE = 42

print("Input:", INPUT_PATH)
print("Target:", TARGET)


## 2. Load and prepare the modeling data

In [ ]:
df = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

target_columns = [
    "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D",
    "Target_DD_10Pct_10D",
]

model_df = df.dropna(subset=[TARGET]).copy()
model_df[TARGET] = model_df[TARGET].astype(int)

excluded = {"Date", *target_columns}

feature_cols = [
    c for c in model_df.columns
    if c not in excluded
    and not c.startswith("Future_")
    and pd.api.types.is_numeric_dtype(model_df[c])
]

X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)
y = model_df[TARGET]

n = len(model_df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train = model_df.iloc[:train_end].copy()
valid = model_df.iloc[train_end:valid_end].copy()
test = model_df.iloc[valid_end:].copy()

X_train = X.iloc[:train_end].copy()
X_valid = X.iloc[train_end:valid_end].copy()
X_test = X.iloc[valid_end:].copy()

y_train = y.iloc[:train_end].copy()
y_valid = y.iloc[train_end:valid_end].copy()
y_test = y.iloc[valid_end:].copy()

print("Shape:", model_df.shape)
print("Features:", len(feature_cols))
print("Date range:", model_df.Date.min(), "->", model_df.Date.max())
print("Train:", train.Date.min(), "->", train.Date.max())
print("Validation:", valid.Date.min(), "->", valid.Date.max())
print("Test:", test.Date.min(), "->", test.Date.max())


## 3. Recreate candidate models

These are the same model families used in Notebook 07 so the comparison is reproducible.


In [ ]:
models = {
    "Logistic_Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ]),
    "Random_Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=6,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),
    "Hist_Gradient_Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=RANDOM_STATE,
        )),
    ]),
}

print(list(models))


## 4. Validation comparison

Model selection is performed on validation data only.

Primary ranking metric:
**PR-AUC (Average Precision)**

Secondary metrics:
- ROC-AUC
- Recall
- Precision
- F1
- Balanced Accuracy
- Brier score


In [ ]:
validation_rows = []
validation_probabilities = {}
validation_models = {}

for name, pipeline in models.items():
    fitted = clone(pipeline)
    fitted.fit(X_train, y_train)

    p = fitted.predict_proba(X_valid)[:, 1]
    validation_probabilities[name] = p
    validation_models[name] = fitted

    pred = (p >= 0.50).astype(int)

    validation_rows.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_valid, p),
        "PR_AUC": average_precision_score(y_valid, p),
        "Precision": precision_score(y_valid, pred, zero_division=0),
        "Recall": recall_score(y_valid, pred, zero_division=0),
        "F1": f1_score(y_valid, pred, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_valid, pred),
        "Brier_Score": brier_score_loss(y_valid, p),
    })

validation_comparison = (
    pd.DataFrame(validation_rows)
    .sort_values("PR_AUC", ascending=False)
    .reset_index(drop=True)
)

display(validation_comparison)


## 5. Compare ranking quality

PR-AUC is important here because the 5% drawdown event is relatively uncommon. A model that ranks high-risk observations correctly is more useful than one that only produces good accuracy from the majority class.


In [ ]:
plt.figure(figsize=(8, 5))

for name, p in validation_probabilities.items():
    precision, recall, _ = precision_recall_curve(y_valid, p)
    plt.plot(recall, precision, label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation Precision-Recall Curves")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))

for name, p in validation_probabilities.items():
    fpr, tpr, _ = roc_curve(y_valid, p)
    auc = roc_auc_score(y_valid, p)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Threshold analysis for every candidate

Instead of assuming 0.50 is optimal, inspect the precision/recall trade-off across practical alert thresholds.

The threshold is still selected from validation data only.


In [ ]:
threshold_grid = np.arange(0.10, 0.91, 0.05)

threshold_rows = []

for name, p in validation_probabilities.items():
    for threshold in threshold_grid:
        pred = (p >= threshold).astype(int)
        threshold_rows.append({
            "Model": name,
            "Threshold": round(float(threshold), 2),
            "Precision": precision_score(y_valid, pred, zero_division=0),
            "Recall": recall_score(y_valid, pred, zero_division=0),
            "F1": f1_score(y_valid, pred, zero_division=0),
            "Balanced_Accuracy": balanced_accuracy_score(y_valid, pred),
            "Alerts": int(pred.sum()),
            "Alert_Rate": float(pred.mean()),
        })

threshold_results = pd.DataFrame(threshold_rows)

best_thresholds = (
    threshold_results
    .sort_values(["Model", "F1"], ascending=[True, False])
    .groupby("Model", as_index=False)
    .head(1)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

display(best_thresholds)


## 7. Practical alert trade-off

For an early-warning system, recall matters because missing a genuine high-risk event can be costly. Precision matters because excessive false alarms reduce usefulness.

We therefore inspect recall and precision together rather than optimizing a single metric blindly.


In [ ]:
plt.figure(figsize=(8, 5))

for name in models:
    sub = threshold_results[threshold_results["Model"] == name]
    plt.plot(sub["Recall"], sub["Precision"], marker="o", label=name)

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Validation Precision–Recall Trade-off by Threshold")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Probability calibration

A risk system should ideally produce probabilities that are meaningful as risk scores.

We inspect validation calibration and Brier score. This is diagnostic only; we do not use the test set for calibration decisions.


In [ ]:
calibration_rows = []

plt.figure(figsize=(8, 5))

for name, p in validation_probabilities.items():
    prob_true, prob_pred = calibration_curve(
        y_valid,
        p,
        n_bins=8,
        strategy="quantile",
    )

    plt.plot(prob_pred, prob_true, marker="o", label=name)

    calibration_rows.append({
        "Model": name,
        "Brier_Score": brier_score_loss(y_valid, p),
        "Validation_Event_Rate": y_valid.mean(),
        "Mean_Predicted_Probability": p.mean(),
    })

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Validation Calibration")
plt.legend()
plt.tight_layout()
plt.show()

calibration_summary = pd.DataFrame(calibration_rows).sort_values("Brier_Score")
display(calibration_summary)


## 9. Stability by validation sub-period

A model can look good on average while failing badly in a particular market regime.

We therefore divide the validation period into chronological blocks and compare PR-AUC across them.


In [ ]:
valid_eval = valid[["Date", TARGET]].copy()
valid_eval["Month"] = valid_eval["Date"].dt.to_period("M")

# Create up to four chronological validation blocks
unique_months = valid_eval["Month"].drop_duplicates().tolist()

if len(unique_months) >= 4:
    block_ids = pd.qcut(
        np.arange(len(unique_months)),
        q=4,
        labels=["V1_Early", "V2", "V3", "V4_Late"],
        duplicates="drop",
    )
    month_to_block = dict(zip(unique_months, block_ids))
    valid_eval["Block"] = valid_eval["Month"].map(month_to_block)
else:
    valid_eval["Block"] = "Validation"

stability_rows = []

for name, p in validation_probabilities.items():
    temp = valid_eval.copy()
    temp["Probability"] = p

    for block, sub in temp.groupby("Block", sort=False):
        if sub[TARGET].nunique() < 2:
            auc = np.nan
            pr_auc = np.nan
        else:
            auc = roc_auc_score(sub[TARGET], sub["Probability"])
            pr_auc = average_precision_score(sub[TARGET], sub["Probability"])

        stability_rows.append({
            "Model": name,
            "Validation_Block": str(block),
            "Rows": len(sub),
            "Event_Rate": sub[TARGET].mean(),
            "ROC_AUC": auc,
            "PR_AUC": pr_auc,
        })

stability = pd.DataFrame(stability_rows)
display(stability)


## 10. Decision rule for the candidate model

We use the following transparent decision rule:

1. Prefer higher validation PR-AUC.
2. Use ROC-AUC as a secondary ranking measure.
3. Prefer reasonable recall/precision trade-offs.
4. Prefer lower Brier score when predictive ranking is similar.
5. Avoid choosing a model solely because it has the highest training performance.

This produces a defensible candidate for Notebook 09 onward.


In [ ]:
ranking = validation_comparison.copy()

ranking["PR_AUC_Rank"] = ranking["PR_AUC"].rank(ascending=False, method="min")
ranking["ROC_AUC_Rank"] = ranking["ROC_AUC"].rank(ascending=False, method="min")
ranking["Brier_Rank"] = ranking["Brier_Score"].rank(ascending=True, method="min")

ranking["Composite_Rank"] = (
    ranking["PR_AUC_Rank"]
    + 0.5 * ranking["ROC_AUC_Rank"]
    + 0.25 * ranking["Brier_Rank"]
)

ranking = ranking.sort_values(
    ["PR_AUC", "Composite_Rank"],
    ascending=[False, True]
).reset_index(drop=True)

display(ranking)

candidate_model = ranking.loc[0, "Model"]

candidate_threshold = float(
    best_thresholds.loc[
        best_thresholds["Model"] == candidate_model,
        "Threshold"
    ].iloc[0]
)

print("Candidate model:", candidate_model)
print("Candidate validation threshold:", candidate_threshold)


## 11. Compare against the Notebook 07 test result

This section is **descriptive only**. We do not change the candidate based on test performance.

It shows the previously locked Notebook 07 test result alongside the selected candidate for transparency.


In [ ]:
test_report_path = REPORTS / "test_predictions.csv"

if test_report_path.exists():
    test_predictions_report = pd.read_csv(test_report_path, parse_dates=["Date"])

    test_summary = pd.DataFrame([{
        "Notebook_07_Selected_Model": candidate_model,
        "Notebook_07_Selected_Threshold": candidate_threshold,
        "Test_ROC_AUC": roc_auc_score(
            test_predictions_report["Actual_Target_DD_5Pct_10D"],
            test_predictions_report["Predicted_Probability"],
        ),
        "Test_PR_AUC": average_precision_score(
            test_predictions_report["Actual_Target_DD_5Pct_10D"],
            test_predictions_report["Predicted_Probability"],
        ),
        "Test_Observations": len(test_predictions_report),
    }])

    display(test_summary)
else:
    print("Notebook 07 test_predictions.csv not found; skipping descriptive test summary.")


## 12. Refit candidate on Train + Validation for downstream use

The candidate is now fixed using development data only.

This fitted artifact is intended for downstream explainability/final-evaluation notebooks. It is **not** used to change the already reported Notebook 07 test metrics.


In [ ]:
train_valid = pd.concat([train, valid], ignore_index=True)

X_train_valid = train_valid[feature_cols].copy()
X_train_valid = X_train_valid.replace([np.inf, -np.inf], np.nan)
y_train_valid = train_valid[TARGET].copy()

candidate_fitted_model = clone(models[candidate_model])
candidate_fitted_model.fit(X_train_valid, y_train_valid)

print("Candidate refit observations:", len(train_valid))
print("Candidate:", candidate_model)
print("Threshold:", candidate_threshold)


## 13. Save Notebook 08 outputs

In [ ]:
MODEL_SELECTION_PATH = REPORTS / "model_selection_final.csv"
THRESHOLD_PATH = REPORTS / "model_threshold_analysis.csv"
CALIBRATION_PATH = REPORTS / "model_calibration_summary.csv"
STABILITY_PATH = REPORTS / "model_validation_stability.csv"

selection_output = ranking.copy()
selection_output["Candidate_Model"] = selection_output["Model"].eq(candidate_model)
selection_output["Candidate_Threshold"] = np.where(
    selection_output["Model"].eq(candidate_model),
    candidate_threshold,
    np.nan
)

selection_output.to_csv(MODEL_SELECTION_PATH, index=False)
threshold_results.to_csv(THRESHOLD_PATH, index=False)
calibration_summary.to_csv(CALIBRATION_PATH, index=False)
stability.to_csv(STABILITY_PATH, index=False)

joblib.dump(
    {
        "model_name": candidate_model,
        "model": candidate_fitted_model,
        "feature_columns": feature_cols,
        "target": TARGET,
        "threshold": candidate_threshold,
        "selection_rule": "Validation PR-AUC primary; ROC-AUC/Brier and practical threshold trade-off secondary",
        "random_state": RANDOM_STATE,
    },
    MODELS / "08_selected_model.joblib"
)

print("Notebook 08 outputs saved:")
print(MODEL_SELECTION_PATH)
print(THRESHOLD_PATH)
print(CALIBRATION_PATH)
print(STABILITY_PATH)
print(MODELS / "08_selected_model.joblib")


## 14. Final quality checks

Before proceeding:
- candidate comes from validation data
- threshold comes from validation data
- no future/target columns enter features
- no infinite values remain
- chronological ordering is preserved
- selected model artifact exists


In [ ]:
assert train["Date"].max() < valid["Date"].min() < test["Date"].min()

assert not any(
    ("future" in c.lower() or "target" in c.lower())
    for c in feature_cols
)

assert np.isinf(
    X_train_valid.select_dtypes(include=np.number)
).sum().sum() == 0

assert candidate_model in models
assert 0 < candidate_threshold < 1

assert (MODELS / "08_selected_model.joblib").exists()

print("Candidate model:", candidate_model)
print("Candidate threshold:", candidate_threshold)
print("Validation PR-AUC:", float(
    validation_comparison.loc[
        validation_comparison["Model"] == candidate_model, "PR_AUC"
    ].iloc[0]
))
print("FINAL NOTEBOOK 08 QUALITY CHECKS: PASSED")


# Conclusion

Notebook 08 establishes the model-selection decision using development data.

The test-set metrics from Notebook 07 remain the final out-of-sample benchmark and are not used to optimize the candidate.

### Next step
`09_hyperparameter_tuning.ipynb`

The next notebook can tune the selected model using **time-aware validation**, without leaking information from the final test period.
